# Circle Area Analysis Workflow - ArcGIS Pro Python Notebook (Adapted for Geodatabases)

**Note:** This notebook has been adapted to work with geodatabase feature classes instead of shapefiles. All changes are documented in `adaptation_notes.md`.

In [ ]:
# ============================================================================
# Phase 2.1: arcpy.mp Integration - Initialize ArcGIS Project
# ============================================================================

import arcpy

# Initialize ArcGIS Pro project using CURRENT keyword
# This works when running from within ArcGIS Pro Python window or Script Tool
aprx = arcpy.mp.ArcGISProject('CURRENT')

# Get the geodatabase path from the project's default workspace
gdb_path = aprx.workspacePath

print(f"ArcGIS Project initialized: {aprx.name}")
print(f"Geodatabase path: {gdb_path}")

In [ ]:
# ============================================================================
# Phase 1.1 & 1.2: Create Point Layers from CSV Tables (Geodatabase Feature Classes)
# ============================================================================

# Input data paths
areas_csv = r"C:\data\areas.csv"
cities_csv = r"C:\data\cities.csv"

# Create point layer from areas CSV (output as feature class, no .shp extension)
arcpy.management.XYTableToPoint(
    input_table=areas_csv,
    output_feature_class=f"{gdb_path}/areas",  # Feature class in geodatabase
    x_field="longitude",
    y_field="latitude",
    coordinate_system="WGS_1984"  # Use proper CRS name instead of numeric code
)

# Create point layer from cities CSV (output as feature class, no .shp extension)
arcpy.management.XYTableToPoint(
    input_table=cities_csv,
    output_feature_class=f"{gdb_path}/cities",  # Feature class in geodatabase
    x_field="longitude",
    y_field="latitude",
    coordinate_system="WGS_1984"
)

In [ ]:
# ============================================================================
# Phase 1.3: Buffer Areas (Geodatabase Feature Classes)
# ============================================================================

arcpy.analysis.Buffer(
    in_features=f"{gdb_path}/areas",  # Feature class, not .shp
    out_feature_class=f"{gdb_path}/area_buffers",  # No .shp extension
    distance=100,  # Use numeric value directly (not arcpy.Field())
    dissolve_option="NONE",     # Keep one buffer per row
    method="PLANAR"             # Automatic geodesic for geographic input
)

In [ ]:
# ============================================================================
# Phase 1.4: Summarize Within (Geodatabase Feature Classes)
# ============================================================================

summary_fields = [["population", "Sum"]]

arcpy.analysis.SummarizeWithin(
    in_polygons=f"{gdb_path}/area_buffers",  # Feature class (not .shp)
    in_summary_features=f"{gdb_path}/cities",  # Feature class (not path with .shp)
    out_feature_class=f"{gdb_path}/summary",  # Feature class output, not CSV
    summary_fields=summary_fields,
    keep_all_input_polygons=True
)

# ============================================================================
# Phase 3.2: Validation - Check Feature Class Properties
# ============================================================================

import arcpy

# Verify feature classes were created successfully
for fc in [f"{gdb_path}/areas", f"{gdb_path}/cities"]:
    describe = arcpy.Describe(fc)
    print(f"Feature class: {fc}")
    print(f"  Type: {describe.type}")
    print(f"  Spatial Reference: {describe.spatialReference.name}")
    print(f"  Is Broken: {describe.isBroken}")

# Verify layer properties in map
aprx = arcpy.mp.ArcGISProject('CURRENT')
m = aprx.listMaps("Circle Analysis")[0]
for lyr in m.listLayers():
    print(f"Layer: {lyr.name}")
    print(f"  Is Feature Layer: {lyr.isFeatureLayer}")
    print(f"  Is Broken: {lyr.isBroken}")

# ============================================================================
# Key Points (Adapted)

- **All output is stored in the geodatabase** from your ArcGIS Pro project
- **Feature classes use no .shp extension** - they are native GDB format
- **Coordinate system uses "WGS_1984"** instead of numeric code 4326
- **Buffer distance is a numeric value (100)** not arcpy.Field()
- **SummarizeWithin outputs a feature class**, not CSV
- **All inputs are geodatabase feature classes**, not shapefiles or CSVs
- **Run from within ArcGIS Pro** using the Python window for CURRENT keyword to work